In [36]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "ebel2019innovative")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Ebel_et al_2019_exp2_10071_2019_1275_MOESM1_ESM_evapecognition.csv")
complete_path_2 = os.path.join(original_data_pathway, "Ebel_et al_2019_exp3_10071_2019_1275_MOESM1_ESM_evapecognition.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [37]:
import pandas as pd
import numpy as np
import pyreadstat

df1 = pd.read_csv(complete_path_1)
df2 = pd.read_csv(complete_path_2)

df1['experiment']="2"
# df1['drop_out']=[]
df2['experiment']="3"
df1['condition'].replace('transparent', 'baseline', inplace=True, regex=True)
# df1['condition'].unique()

In [38]:

data_frames=[df1, df2]

for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s)
    x = x.rename(columns={"subject": "ape",
        "group": "group_original",
        "date": "date_original"})
    x['study_id']="ebel2019innovative"
    data_frames[index]=x
new_df=data_frames[0]


In [39]:
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)

In [40]:
fulldf[['group_temp','group']] = fulldf['group_original'].str.split('-',expand=True)


In [41]:
fulldf[['day','month', 'month2']] = fulldf['date_original'].str.split('.',expand=True)

date_replace = [['jan', '01'],
                ['may', '05'],
                ['jun', '06'],
                ['jul', '07'],
                ['sep', '09'],
                ['oct', '10'],
                ['nov', '11'],
                ['dec', '12']]
for x,y in date_replace:
    fulldf['month'] = fulldf['month'].str.replace(x, y)

In [42]:
year=[]

for index, row in fulldf.iterrows():
    # print(type(row['month']))
    if row['month'].strip() == '01':
        year.append("2015")
    else:
        year.append("2014")
fulldf = fulldf.assign(year=year)


In [43]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
fulldf['ape'] = fulldf['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
fulldf= fulldf.merge(apedf,left_on='ape', right_on='name', how='left', suffixes=('_original', ''))


In [44]:
remdf=["toba", "tao"]
fulldf = fulldf[~fulldf.ape.isin(remdf)]

In [45]:
import re
replace_1=re.compile('(\-|\.)') 
replace_2=re.compile('(\;|\:|\,)') 
fulldf.columns = fulldf.columns.str.replace(replace_1, '_')

character_replace = [
                    # ['solution', 0, np.nan],
                    ['latency_solution', 0, np.nan], 
                    ['comment', replace_2, ''],
                    ['tested_before', '\.', '_'],
                    ['test_phase', '\.', '_']]
for x,y,k in character_replace:
    fulldf[x].replace(y, k, inplace=True, regex=True)

fulldf['solution'] = fulldf['solution'].astype("Int64")


In [46]:
# fulldf.columns
fulldf.rename(columns={"ape": "participant", "group":"species_subgroup",
                       'age':'age_original'}, inplace=True)


In [47]:
comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 
fulldf= fulldf.merge(ape_dob,left_on='participant', right_on='name', how='left') 
fulldf['dodc'] = fulldf['year'].astype(str) + '-' + fulldf['month'].astype(str) + '-' + fulldf['day'].astype(str)
fulldf['dodc'] = pd.to_datetime(fulldf['dodc'])
fulldf['dob'] = pd.to_datetime(fulldf['dob'])

fulldf['age_in_years'] = (fulldf['dodc'] - fulldf['dob']).dt.days//365

fulldf['age_in_years'] = fulldf['age_in_years'].astype("Int64")

In [48]:
group_addition = ['frodo', 'lome','tai']

for x in group_addition:
    fulldf.loc[fulldf.participant == x, ['species_subgroup']] = 'a'



fulldf['condition'].replace('apeaction_test', 'water_tap_by_ape', inplace=True, regex=True)
fulldf['condition'].replace('humanaction_test', 'water_tap_by_human', inplace=True, regex=True)
# fulldf['condition'].unique()

In [49]:

fulldf=fulldf[['study_id',  'experiment', 'year', 'month', 'day', 
       'participant',   'age_original','age_in_years', 'sex','species','species_subgroup', 'session','condition',
        'tested_before', 'food_reward', 'test_phase', 
        'solution', 'latency_solution', 'number_spits',
       'latency_firstspit', 'mean_inter_spit_interval', 'comment'  ]]


In [50]:
for index in range(2,4):
    exp = fulldf[fulldf['experiment'] == str(index)]
    exp = exp.dropna(axis=1, how='all')
    comp_out_path = os.path.join(out_pathway, 'ebel2019innovative_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway, 'ebel2019innovative_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)